# ⚖️ PakLex AI — Notebook 1: PDF Ingestion & Vector Store

**Purpose:** Load PPC PDF from `../data/` folder, chunk with LangChain, embed with `all-MiniLM-L6-v2`, and persist to ChromaDB.

---
| Step | Task |
|------|------|
| 1 | Install dependencies |
| 2 | Load PDF using LangChain `PyPDFLoader` |
| 3 | Clean & inspect the loaded text |
| 4 | Chunk with LangChain `RecursiveCharacterTextSplitter` (512 tokens / 50 overlap) |
| 5 | Embed with `HuggingFaceEmbeddings` (MiniLM-L6-v2) |
| 6 | Store in ChromaDB via LangChain `Chroma` |
| 7 | Verify with test queries |

---
> 📂 **Place your PPC PDF** inside the `../data/` folder before running.
> The notebook auto-detects any `.pdf` file in that folder.

## Step 1 — Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install -q \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    chromadb \
    sentence-transformers \
    python-dotenv \
    pypdf \
    groq \
    tiktoken \
    tqdm

print('✅ All packages installed successfully')

✅ All packages installed successfully



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Configuration

In [2]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv

# ── Load .env (API key lives there — never hardcoded) ──────────────────────
load_dotenv('../.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError(
        '❌ GROQ_API_KEY not found!\n'
        '   1. Get free key at https://console.groq.com\n'
        '   2. Create ../.env file\n'
        '   3. Add: GROQ_API_KEY=gsk_your_key_here'
    )
print(f'✅ Groq API key loaded: gsk_...{GROQ_API_KEY[-6:]}')

# ── Project paths ──────────────────────────────────────────────────────────
DATA_DIR         = Path('../data')
VECTOR_STORE_DIR = '../vector_store'
COLLECTION_NAME  = 'pakistan_penal_code'
EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'

# ── Chunking config ────────────────────────────────────────────────────────
CHUNK_SIZE    = 1000   # characters (LangChain uses chars not tokens)
CHUNK_OVERLAP = 200    # characters overlap

# Create dirs
DATA_DIR.mkdir(parents=True, exist_ok=True)
Path(VECTOR_STORE_DIR).mkdir(parents=True, exist_ok=True)

print(f'✅ Config ready')
print(f'   Data folder  : {DATA_DIR.resolve()}')
print(f'   Vector store : {VECTOR_STORE_DIR}')
print(f'   Chunk size   : {CHUNK_SIZE} chars / {CHUNK_OVERLAP} overlap')

✅ Groq API key loaded: gsk_...Xwtfbg
✅ Config ready
   Data folder  : C:\Users\Umair_Anjum\Downloads\Rag_Project\data
   Vector store : ../vector_store
   Chunk size   : 1000 chars / 200 overlap


## Step 3 — Detect & Load PDF with LangChain

In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# ── Auto-detect PDF files in data/ folder ─────────────────────────────────
pdf_files = list(DATA_DIR.glob('*.pdf'))

if not pdf_files:
    raise FileNotFoundError(
        f'❌ No PDF found in {DATA_DIR.resolve()}\n'
        f'   Place your PPC PDF file in the data/ folder and re-run.'
    )

print(f'📂 Found {len(pdf_files)} PDF file(s) in data/:')
for f in pdf_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f'   📄 {f.name}  ({size_mb:.2f} MB)')

# ── Load all PDFs using LangChain ─────────────────────────────────────────
print('\n⏳ Loading PDF(s) with LangChain PyPDFLoader...')
all_pages = []

for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages  = loader.load()           # returns list of Document objects
    
    # Add source filename to metadata
    for page in pages:
        page.metadata['source_file'] = pdf_path.name
    
    all_pages.extend(pages)
    print(f'   ✓ {pdf_path.name}: {len(pages)} pages loaded')

print(f'\n✅ Total pages loaded : {len(all_pages)}')
print(f'\nSample page metadata:')
print(f'   {all_pages[0].metadata}')
print(f'\nSample text (first 300 chars):')
print(f'   {all_pages[0].page_content[:300]}...')

📂 Found 1 PDF file(s) in data/:
   📄 Pak_penal_code.pdf  (1.54 MB)

⏳ Loading PDF(s) with LangChain PyPDFLoader...


   ✓ Pak_penal_code.pdf: 179 pages loaded

✅ Total pages loaded : 179

Sample page metadata:
   {'source': '..\\data\\Pak_penal_code.pdf', 'page': 0, 'source_file': 'Pak_penal_code.pdf'}

Sample text (first 300 chars):
    
Page 1 of 179 
  
 
 
 
THE PAKISTAN PENAL CODE  
 
 
 
 
 
 
CONTENTS  
 
CHAPTER I  
INTRODUCTION  
1. Title and extent of operation of the Code  
2. Punishment of offences committed within Pakistan  
3. Punishment of offences committed beyond, but which by law may be tried within, Pakist an 
4....


## Step 4 — Clean Text

In [4]:
import re
from langchain.schema import Document

def clean_page(doc: Document) -> Document:
    """
    Clean PDF extracted text:
    - Remove excessive whitespace
    - Remove page headers/footers (page numbers, repeated titles)
    - Fix hyphenation across lines
    """
    text = doc.page_content
    
    # Fix hyphenated line breaks (e.g. 'imprison-\nment' → 'imprisonment')
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    
    # Replace multiple newlines with double newline
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    # Remove lone page numbers (e.g. lines with only digits)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    
    # Collapse multiple spaces
    text = re.sub(r'  +', ' ', text)
    
    doc.page_content = text.strip()
    return doc

# Apply cleaning
cleaned_pages = [clean_page(p) for p in all_pages if len(p.page_content.strip()) > 50]

print(f'✅ Cleaning complete')
print(f'   Pages before : {len(all_pages)}')
print(f'   Pages after  : {len(cleaned_pages)} (removed empty/short pages)')

# Total character count
total_chars = sum(len(p.page_content) for p in cleaned_pages)
print(f'   Total chars  : {total_chars:,}')

✅ Cleaning complete
   Pages before : 179
   Pages after  : 179 (removed empty/short pages)
   Total chars  : 506,550


## Step 5 — Chunk with LangChain RecursiveCharacterTextSplitter

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── RecursiveCharacterTextSplitter ─────────────────────────────────────────
# Why RecursiveCharacterTextSplitter?
# Splits on paragraph → sentence → word boundaries in order.
# For legal text, this preserves complete clauses better than naive splitting.

splitter = RecursiveCharacterTextSplitter(
    chunk_size      = CHUNK_SIZE,    # 1000 chars ≈ 200-250 tokens
    chunk_overlap   = CHUNK_OVERLAP, # 200 chars overlap = ~50 tokens
    length_function = len,
    separators      = [
        '\n\n',   # paragraph break (highest priority)
        '\n',     # line break
        '. ',     # sentence end
        ', ',     # clause boundary
        ' ',      # word boundary
        '',       # character (last resort)
    ]
)

# Split all cleaned pages into chunks
chunks = splitter.split_documents(cleaned_pages)

print(f'✅ Chunking complete')
print(f'   Input pages  : {len(cleaned_pages)}')
print(f'   Output chunks: {len(chunks)}')
print(f'   Avg chunk len: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars')

print(f'\nSample chunk [5]:')
print(f'   Metadata : {chunks[5].metadata}')
print(f'   Text     : {chunks[5].page_content[:250]}...')

✅ Chunking complete
   Input pages  : 179
   Output chunks: 676
   Avg chunk len: 859 chars

Sample chunk [5]:
   Metadata : {'source': '..\\data\\Pak_penal_code.pdf', 'page': 4, 'source_file': 'Pak_penal_code.pdf'}
   Text     : Page 5 of 179 
 88. Act not intended to cause death, done by consent in g ood faith for person's benefit 
89. Act done in good faith for benefit of child or insane person, by or by consent of guardian 
90. Consent known to be given under fear or misc...


## Step 6 — Embed with LangChain HuggingFaceEmbeddings

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

print(f'⏳ Loading embedding model: {EMBEDDING_MODEL}')
print('   (First run downloads ~90MB — cached afterwards)')

embeddings = HuggingFaceEmbeddings(
    model_name      = EMBEDDING_MODEL,
    model_kwargs    = {'device': 'cpu'},
    encode_kwargs   = {'normalize_embeddings': True},  # needed for cosine similarity
)

# Quick test
test_vec = embeddings.embed_query('What is the punishment for murder?')

print(f'✅ Embedding model ready')
print(f'   Model      : {EMBEDDING_MODEL}')
print(f'   Dimensions : {len(test_vec)}')
print(f'   Device     : CPU')

⏳ Loading embedding model: all-MiniLM-L6-v2
   (First run downloads ~90MB — cached afterwards)


C:\Users\Umair_Anjum\AppData\Local\Programs\Python\Python311\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Embedding model ready
   Model      : all-MiniLM-L6-v2
   Dimensions : 384
   Device     : CPU


## Step 7 — Store in ChromaDB via LangChain

In [7]:
from langchain_community.vectorstores import Chroma
from tqdm.notebook import tqdm

print(f'⏳ Building ChromaDB vector store...')
print(f'   Chunks to embed & store: {len(chunks)}')
print(f'   This may take 1-3 minutes...')

# LangChain Chroma handles embedding + storage in one call
# Processes in batches automatically
vectordb = Chroma.from_documents(
    documents          = chunks,
    embedding          = embeddings,
    persist_directory  = VECTOR_STORE_DIR,
    collection_name    = COLLECTION_NAME,
    collection_metadata= {'hnsw:space': 'cosine'},
)

# Persist to disk
vectordb.persist()

doc_count = vectordb._collection.count()
print(f'\n✅ ChromaDB vector store built and persisted')
print(f'   Location   : {VECTOR_STORE_DIR}/')
print(f'   Collection : {COLLECTION_NAME}')
print(f'   Stored     : {doc_count} chunks')

⏳ Building ChromaDB vector store...
   Chunks to embed & store: 676
   This may take 1-3 minutes...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



✅ ChromaDB vector store built and persisted
   Location   : ../vector_store/
   Collection : pakistan_penal_code
   Stored     : 676 chunks


C:\Users\Umair_Anjum\AppData\Local\Temp\ipykernel_10424\2344229394.py:19: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


## Step 8 — Verify with Test Queries

In [8]:
# Create a retriever from the vector store
retriever = vectordb.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 5}
)

def test_retrieval(query: str):
    docs = retriever.get_relevant_documents(query)
    print(f'\n🔍 Query: "{query}"')
    print('─' * 60)
    for i, d in enumerate(docs):
        print(f'  [{i+1}] Page {d.metadata.get("page", "?")} | {d.metadata.get("source_file", "")} ')
        print(f'       {d.page_content[:180]}...')

test_retrieval('What is the punishment for murder Qatl-i-amd?')
test_retrieval('Define theft and punishment for robbery')
test_retrieval('What is sedition under Pakistan law?')

C:\Users\Umair_Anjum\AppData\Local\Temp\ipykernel_10424\3961606066.py:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents(query)
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔍 Query: "What is the punishment for murder Qatl-i-amd?"
────────────────────────────────────────────────────────────
  [1] Page 16 | Pak_penal_code.pdf 
       Page 17 of 179 
 311. Ta'zir after waiver or compounding of right of qisas in qatl­i­amd 
312. Qatl­i­'amd after waiver or c ompounding of qisas 
313. Right of qisas in qatl­i­amd ...
  [2] Page 105 | Pak_penal_code.pdf 
       301. Causing d eath of person other than the person whose death was intended. Where 
a person, by doing anything which he intends or knows to be likely to cause death, causes death...
  [3] Page 110 | Pak_penal_code.pdf 
       ordinary course of nature is not likely to cause death is said to commit qatl­shibh­i­ ’amd. 
Illustration 
A in order to cause hurt strikes Z with a stick or stone which in the or...
  [4] Page 110 | Pak_penal_code.pdf 
       Page 111 of 179 
 court, an officer authorised by the court shall give permission for the execution of qisas and the 
Government shall cause execution of 

## ✅ Ingestion Complete!

```
vector_store/
└── chroma.sqlite3   ← persisted to disk
```

**Next:** Open `02_rag_engine.ipynb`

> Make sure your `.env` file has your Groq API key:
> ```
> GROQ_API_KEY=gsk_xxxxxxxxxxxxxxxxxxxx
> ```